In [1]:
import os
import torch
from tqdm import tqdm

In [2]:
MODELS_DIR = "models"
DATASET_DIR = "dataset"

In [3]:
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

using device: cuda


In [5]:
print("bf16 supported:",torch.cuda.is_bf16_supported())

bf16 supported: True


# Dataset

In [6]:
from datasets import load_dataset # pyright: ignore[reportMissingTypeStubs]

raw_forget_ds = load_dataset(
    "locuslab/TOFU",
    "forget01",
    cache_dir=DATASET_DIR,
)["train"]

raw_retain_ds = load_dataset(
    "locuslab/TOFU",
    "retain99",
    cache_dir=DATASET_DIR
)["train"]

/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Model

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=MODELS_DIR)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    cache_dir=MODELS_DIR,
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1374.06it/s]


In [8]:
print(tokenizer.pad_token)
print(tokenizer.eos_token)
print(tokenizer.bos_token)
print(model.dtype)

<|endoftext|>
<|endoftext|>
None
torch.bfloat16


In [9]:
tokenizer.special_tokens_map

{'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}

In [10]:
def print_trainable_parameters(model):
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )
print_trainable_parameters(model)

trainable params: 494032768 || all params: 494032768 || trainable%: 100.0


In [11]:
from datasets import concatenate_datasets # pyright: ignore[reportMissingTypeStubs]

train_ds = concatenate_datasets([
    raw_retain_ds,
    raw_forget_ds,
])

In [12]:
def formatting_func(example):
    return {
        "prompt": [
            {
                "role": "user",
                "content": example["question"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["answer"],
            },
        ],
    }  # type: ignore


train_ds = train_ds.map(
    formatting_func,
    remove_columns=train_ds.column_names,
)

Map: 100%|██████████| 4000/4000 [00:00<00:00, 35973.43 examples/s]


In [16]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=os.path.join(MODELS_DIR, "checkpoints_trl"),
    num_train_epochs=50,
    max_steps=-1,
    learning_rate=1e-4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    bf16=True,
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    max_length=512,
    packing=False,
    completion_only_loss=True,
    save_total_limit=2,
)

In [17]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    processing_class=tokenizer,
)

Tokenizing train dataset: 100%|██████████| 4000/4000 [00:01<00:00, 3106.58 examples/s]


In [18]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,3.937883
10,2.874231
15,2.593794
20,2.298833
25,2.457640
30,2.232765
35,2.285957
40,2.055854
45,2.091041
50,2.291351


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.29s/it]


KeyboardInterrupt: 

In [19]:
trainer.save_model(os.path.join(MODELS_DIR, "final_model_trainer_trl"))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.54s/it]


# Eval

In [20]:
@torch.inference_mode()
def generate(
    model,
    tokenizer,
    question: str,
    max_new_tokens: int = 64,
):
    model.eval()
    messages = [
        {"role": "user", "content": question}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)


    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

In [21]:
from eval_utils import Config, ForgettingLLMJudge
config = Config.from_dotenv()
judge = ForgettingLLMJudge(openai_config=config.openai)

# Eval Full Model on Forget Set

In [ ]:
results = []

for sample in tqdm(raw_forget_ds):

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

100%|██████████| 40/40 [03:16<00:00,  4.91s/it]


In [23]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 8.525,
 'avg_forgetting': 0.7,
 'avg_hallucination': 2.45,
 'retention_rate': 0.7,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.05}

# Eval Full Model on Retain Set's Sample

In [28]:
import random
results = []

ids: set[int] = set()
for _ in tqdm(range(100)):
    id = random.randint(0, len(raw_retain_ds) - 1)
    while id in ids:
        id = random.randint(0, len(raw_retain_ds) - 1)
    ids.add(id)
    sample = raw_retain_ds[id]

    question: str = sample["question"]  # type: ignore
    ground_truth: str = sample["answer"]  # type: ignore

    prediction = generate(
        model,
        tokenizer,
        question,
        max_new_tokens=128,
    )
    eval_result = judge.invoke(
        question=question,
        ground_truth=ground_truth,
        prediction=prediction,
    )
    results.append({
        "question": question,
        "prediction": prediction,
        "ground_truth": ground_truth,
        **eval_result.model_dump()
    })

 80%|████████  | 80/100 [07:23<02:12,  6.63s/it]/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/llm-unlearning/eval_utils/llm.py:35: UserWarning: error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jnqr6ta3f5g8jnakgaxawzzy` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199088, Requested 1331. Please try again in 3m1.008s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
  warnings.warn(f"error: {e}")
 81%|████████  | 81/100 [07:29<02:04,  6.57s/it]/home/a8c3b742-5df0-8324-baef/local-development/machine-unlearning/llm-unlearning/eval_utils/llm.py:35: UserWarning: error: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jnqr6ta3f5g8jnakgaxawzzy` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 199072, Re

In [29]:
import pandas as pd

df = pd.DataFrame(results)

metrics = {
    "avg_correctness":
        float(df.correctness_score.mean()),

    "avg_forgetting":
        float(df.forgetting_score.mean()),

    "avg_hallucination":
        float(df.hallucination_score.mean()),

    "retention_rate":
        float((df.verdict == "retained").mean()),

    "forgetting_rate":
        float((df.verdict == "forgotten").mean()),

    "hallucination_rate":
        float((df.verdict == "hallucinated").mean()),
}

metrics

{'avg_correctness': 8.74,
 'avg_forgetting': 1.71,
 'avg_hallucination': 3.14,
 'retention_rate': 0.7,
 'forgetting_rate': 0.0,
 'hallucination_rate': 0.06}